In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
cd /content/gdrive/My Drive/Colab Notebooks/

/content/gdrive/My Drive/Colab Notebooks


# **Install Dependencies**

In [3]:
!pip install -q PyPDF2 sentence-transformers faiss-cpu transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 102.8 MB/s eta 0:00:00


# **Imports**

Imports the previously installed modules into the active Python environment. This includes numpy, faiss, PyPDF2, and necessary classes from the transformers and sentence_transformers libraries.

In [4]:
import textwrap
import numpy as np
import faiss
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# **Uploading PDF**

In [5]:
PDF_PATH = "Akshita Resume.pdf"

# **Document Ingestion**

Defines and runs a load_pdf function that reads the specified PDF file page by page. It extracts all the raw text into a single string and prints the total character count (4,786 characters).

In [6]:
def load_pdf(path: str) -> str:
    reader = PdfReader(path)
    raw_text = ""
    for page in reader.pages:
        text = page.extract_text()
        if text:
            raw_text += text + "\n"
    return raw_text

document_text = load_pdf(PDF_PATH)
print(f"Extracted {len(document_text)} characters from {PDF_PATH}")

Extracted 4786 characters from Akshita Resume.pdf


# **Text Chunking**

Defines a chunk_text function to slice the massive text string into 500-character pieces with a 50-character overlap. It applies this function and confirms the document was split into 11 chunks.

In [7]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(document_text, chunk_size=500, overlap=50)
print(f"Document split into {len(chunks)} chunks")

Document split into 11 chunks


# **Embedding Creation**

Initializes the all-MiniLM-L6-v2 model from Hugging Face. It then encodes the 11 text chunks into mathematical vectors, resulting in an array of embeddings.

In [8]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


def create_embeddings(text_chunks: list[str]) -> np.ndarray:
    return embedding_model.encode(text_chunks, show_progress_bar=True)


chunk_embeddings = create_embeddings(chunks)
print(f"Created embeddings of shape: {chunk_embeddings.shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Created embeddings of shape: (11, 384)


# **Vector Database (FAISS)**

Creates a custom VectorStore class using the FAISS library. It adds the generated embeddings and their corresponding text chunks into this database for high-speed similarity search.

In [9]:
class VectorStore:

    def __init__(self, dimension: int):
        self.index = faiss.IndexFlatL2(dimension)
        self.text_chunks: list[str] = []

    def add(self, embeddings: np.ndarray, texts: list[str]) -> None:
        self.index.add(np.array(embeddings).astype("float32"))
        self.text_chunks.extend(texts)

    def search(self, query_embedding: np.ndarray, top_k: int = 3) -> list[str]:
        distances, indices = self.index.search(
            np.array([query_embedding]).astype("float32"), top_k
        )
        return [self.text_chunks[i] for i in indices[0] if i < len(self.text_chunks)]

vector_store = VectorStore(dimension=chunk_embeddings.shape[1])
vector_store.add(chunk_embeddings, chunks)
print(f"Vector store now contains {len(vector_store.text_chunks)} chunks")

Vector store now contains 11 chunks


# **Query Processing**

Defines an embed_query function. This takes a user's text question and converts it into a vector using the same MiniLM model, allowing it to be compared against the database.

In [10]:
def embed_query(query: str) -> np.ndarray:
    return embedding_model.encode(query)

# **Context Retrieval**

Defines a retrieve_context function that queries the FAISS vector database. It returns the top 3 text chunks whose embeddings most closely match the user's query embedding.

In [11]:
def retrieve_context(query: str, top_k: int = 3) -> list[str]:
    query_embedding = embed_query(query)
    return vector_store.search(query_embedding, top_k=top_k)

# **Answer Generation**

Loads the google/flan-t5-base language model via the transformers pipeline. It defines a function that forces the model to answer the user's question using strictly the retrieved context chunks.

In [12]:
generator = pipeline("text-generation", model="google/flan-t5-base")


def generate_answer(query: str, context_chunks: list[str]) -> str:
    context = "\n".join(context_chunks)
    prompt = (
        "Answer the question using only the context below.\n\n"
        f"Context:\n{context}\n"
    )
    result = generator(prompt, max_length=200, do_sample=False)
    return result[0]["generated_text"]

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'ExaoneMoeForCausal

# **Full RAG Pipeline (end-to-end)**

Combines the retrieval and generation steps into a single, unified rag_pipeline function. It prints the user's question, displays the retrieved context, and outputs the final AI-generated answer.

In [13]:
def rag_pipeline(query: str, top_k: int = 3) -> str:
    print(f"Question: {query}")

    retrieved_chunks = retrieve_context(query, top_k=top_k)
    print("\nRetrieved Context:")
    for i, chunk in enumerate(retrieved_chunks, 1):
        print(f"\n[{i}] {textwrap.shorten(chunk, width=150)}")

    answer = generate_answer(query, retrieved_chunks)
    print(f"Answer: {answer}")
    return answer

# **Sample Questions**

In [14]:
rag_pipeline("What is the candidate's current job title and company?")

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Question: What is the candidate's current job title and company?

Retrieved Context:

[1] growth opportunities.2025 - Present Business Development Executive Industry Exhibitions & Business RepresentationWORK EXPERIENCEPROFILE Business [...]

[2] r Completed a one-month industrial training in the pharmaceutical industry, where I gained hands-on experience in quality control, production [...]

[3] g business leads. Represented Tosc International at PHARMAC South 2026, Chennai, developing industry connections and exploring strategic [...]


[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: Answer the question using only the context below.

Context:
growth opportunities.2025 - Present Business Development Executive
Industry Exhibitions & Business RepresentationWORK EXPERIENCEPROFILE
Business Development professional with a Bachelor's degree in Pharmacy and practical experience in client
relationship management, lead generation, and pharmaceutical business development. Currently pursuing
an MBA from Indira Gandhi National Open University to strengthen managerial and strategic skills, seeking
opportunities to contribute to organizational gro
r
Completed a one-month industrial training in the pharmaceutical industry, where I gained hands-on
experience in quality control, production processes, raw material department, clinical trials and quality
assurance.
Mahila Chikitsalaya, Haldwani
Completed a one-month hospital training where I learned about various aspects of pharmaceutical care,
including medication management, patient counselling, and the role of a pharmacist 

"Answer the question using only the context below.\n\nContext:\ngrowth opportunities.2025 - Present Business Development Executive\nIndustry Exhibitions & Business RepresentationWORK EXPERIENCEPROFILE\nBusiness Development professional with a Bachelor's degree in Pharmacy and practical experience in client\nrelationship management, lead generation, and pharmaceutical business development. Currently pursuing\nan MBA from Indira Gandhi National Open University to strengthen managerial and strategic skills, seeking\nopportunities to contribute to organizational gro\nr\nCompleted a one-month industrial training in the pharmaceutical industry, where I gained hands-on\nexperience in quality control, production processes, raw material department, clinical trials and quality\nassurance.\nMahila Chikitsalaya, Haldwani\nCompleted a one-month hospital training where I learned about various aspects of pharmaceutical care,\nincluding medication management, patient counselling, and the role of a pha

In [15]:
rag_pipeline("Which trade exhibitions has the candidate represented the company at?")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Which trade exhibitions has the candidate represented the company at?

Retrieved Context:

[1] g business leads. Represented Tosc International at PHARMAC South 2026, Chennai, developing industry connections and exploring strategic [...]

[2] opportunities to contribute to organizational growth through effective business expansion and stakeholder engagement.Akshita Bora Tosc [...]

[3] s.July 2024 January 2023 ACHIEVEMENTS Successfully managed relationships with B2B clients, including doctors, healthcare professionals, and [...]
Answer: Answer the question using only the context below.

Context:
g business leads.
Represented Tosc International at PHARMAC South 2026, Chennai, developing industry connections
and exploring strategic business opportunities.
Participated in the Indian Pharma Fair, Lucknow, promoting the company's pharmaceutical portfolio
and interacting with potential clients and industry stakeholders.
Generated and nurtured business leads obtained through pharmac

"Answer the question using only the context below.\n\nContext:\ng business leads.\nRepresented Tosc International at PHARMAC South 2026, Chennai, developing industry connections\nand exploring strategic business opportunities.\nParticipated in the Indian Pharma Fair, Lucknow, promoting the company's pharmaceutical portfolio\nand interacting with potential clients and industry stakeholders.\nGenerated and nurtured business leads obtained through pharmaceutical trade fairs, contributing to\nthe company's market expansion initiatives.\nEnhanced brand visibility and st\nopportunities to contribute to organizational growth through effective business expansion and stakeholder\nengagement.Akshita Bora\nTosc International Pvt. Ltd.\nRepresented the company as an Exhibitor at the iPHEX 2025, New Delhi, engaging with domestic\nand international pharmaceutical companies, dermatologists, and healthcare industry professionals.\nParticipated in India Pharma Expo 2026, Hyderabad, showcasing the compa